# PRÁCTICA 1

---


In [ ]:
import cv2, numpy as np, matplotlib.pyplot as plt
import os
import imageio
import numpy as np
from typing import List
from utils import get_hsv_color_ranges
import glob

In [ ]:
img = cv2.imread('../data/003.jpg')  # la función de imread() toma el path de la imagen como argumento

In [ ]:
cv2.imshow("First image", img)  # a imshow() tengo que pasarle los argumentos del nombre que quiero darle a la imagen, y la propia imagen (Mat, de matriz)
cv2.waitKey(0)  # en waitKey() ponemos la tecla del teclado con la que queremos cerrar la imagen
cv2.destroyAllWindows() # esto cierra todas las ventanas de despliegue de imágenes que hayamos abierto antes

In [ ]:
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)  # cvtColor toma la imagen y el código de color al que quieres convertir la imagen (en este caso a grises)

print("Color shape:", img.shape)  # el atributo shape muestra el alto, ancho y el número de canales de color de los que dispongo
print("Gray shape:", gray.shape)  # en la shape de la imagen en gris solo mostrará 2 dimensiones: la altura y el ancho, ya que el canal ahora ya es solo 1 en vez de 3

"""
En Python/NumPy, el atributo .shape devuelve las dimensiones en el siguiente orden:

    Índice 0: Alto (Height) -> Número de filas de píxeles.

    Índice 1: Ancho (Width) -> Número de columnas de píxeles.

    Índice 2: Canales (Channels) -> Solo si la imagen es a color (ej. 3 para BGR).
"""


In [ ]:
new_height_size, new_width_size = 200, 200
resized = cv2.resize(src=img, dsize=(new_height_size, new_width_size) )  
# le tengo que pasar la imagen (matriz) y en una tupla, el tamaño al que lo quiero redimensionar

In [ ]:
# Crop the center of the image 'img' with an approximate size of 120x160 pixels.
h, w = img.shape[:2]  # calculamos cuánto vale el total de la altura y anchura de la imagen
cy, cx = h//2, w//2  # tomamos la mitad de la altura y la mitad de la anchura, y esa tupla de puntos nos define el centro de la imagen
new_h, new_w = 120, 160  # definimos cuánto queremos que mida la imagen recortada, el ROI
half_of_new_h, half_of_new_w = new_h//2, new_w//2  # calculamos cuánto tenemos que sumar y restar desde el centro para crear nuestro recorte

# Definimos los límites de las nuevas filas y columnas de la matriz recortada
new_top_row, new_down_row = cy-half_of_new_h, cy+half_of_new_h
new_left_col, new_right_col = cx-half_of_new_w, cx+half_of_new_w

# Recortamos la imagen
roi = img[new_top_row:new_down_row, new_left_col:new_right_col]  # slicing normal de matrices 
# (tener en cuenta filas, columnas de arriba a abajo)

In [ ]:
# Rotate around the center using cv2.getRotationMatrix2D and cv2.warpAffine.

M = cv2.getRotationMatrix2D(center=(cy,cx), angle=90, scale=1)  # los ángulos hay que pasarlos en grados (positivos significa sentido antihorario)
# la función de getRotationMatrix2D, calcula la matriz de rotación para una imagen en 2 D

print(M) 

rotated = cv2.warpAffine(src=img, M=M, dsize=img.shape[:2])
# la función warpAffine aplica una transformación afín a una imagen (en este caso un giro)

# TODO: Display the image using cv2.imshow().
cv2.imshow("Rotated image", rotated)
cv2.waitKey(0)
cv2.destroyAllWindows()

In [ ]:
# Save the image to disk as 'salida_opcionA_1.png' using cv2.imwrite().
save_img_path = "salida_opcionA_1.png"
cv2.imwrite(save_img_path, img)  # a la función de imwrite hay que pasarle el path donde queremos guardar la imagen, y la propia matriz de la imagen

In [ ]:
import glob  # librería necesaria para buscar nombres de paths con patrones concretos, usando *, etc.

def load_images(filenames: List) -> List:
    return [cv2.imread(filename) for filename in filenames]

# Build a list containing the paths of all images in the data folder
imgs_gen_path = '../data/*.jpg' # patrón que cumplen todos los paths de los archivos de imágenes
imgs_path = sorted(glob.glob(imgs_gen_path))  # cogemos todos los paths que cumplen ese patrón genérico (usando glob) y los ordenamos (usando sorted)
imgs = load_images(imgs_path)  # le pasamos a load_images la lista de paths o filenames de las imágenes que queremos cargar

print(f"Paths de las imágenes: {imgs_path}")
print(f"Número de imágenes cargadas: {len(imgs)}")


In [ ]:
def show_image(img: np.array, img_name: str = "Image"):
    cv2.imshow(img_name, img)  # mostramos la imagen, poniendole el nombre correspondiente a la ventana desplegada
    cv2.waitKey(0)  # ponemos una tecla de introducción
    cv2.destroyAllWindows()

def write_image(output_folder: str, img_name: str, img: np.array):
    img_path = os.path.join(output_folder, img_name)  # creamos el path donde vamos a guardar la imagen
    os.makedirs(output_folder, exist_ok=True)
    cv2.imwrite(img_path, img)  # guardamos la imagen en su correspondiente ubicación

In [ ]:
# Get a list with ìmgs in HSV color space
hsv_imgs = [cv2.cvtColor(im, cv2.COLOR_BGR2HSV) for im in imgs]  

In [ ]:
light_orange = (0, 200, 120)  # son los del min
dark_orange = (255, 255, 255)   # son los del max

# Compute a list that contains a mask (which should segment orange colors) for every image.
orange_masks = [cv2.inRange(hsv_im, light_orange, dark_orange) for hsv_im in hsv_imgs]

"""

La función de inRange lo que hace es comprobar si cada pixel del primer array que le pasas, 
cumple que sus valores de intensidad (color) están en el rango
que delimitan los otros 2 arrays que le pasas. Hace la verificación para los 3 canales (HSV).

Devuelve 255 (True) o 0 (False), en función de si el pixel está en el rango o no, y así se crea la máscara booleana. 
La estructura que devuelve cada vez, es de la misma altura y anchura que la imagen que se le había pasado (matriz)

"""

# Compute a list that contains the result of multiplying the original image with its orange colors mask.
orange_segmented = [cv2.bitwise_and(img, img, mask=mask)
                        for img, mask in zip(imgs, orange_masks)]

"""

La función de bitwise_and lo que hace es hacer la conjunción/intersección pixel a pixel, del pixel de img con el pixel de img,
es decir de un mismo pixel consigo mismo, pero lo hace solo en el caso donde la máscara en ese bit sea True.

Entonces como cada pixel se representa por una serie de bits, la intersección de un pixel consigo mismo da siempre el mismo pixel (los mismos bits) pero ese
resultado solo se pone cuando la mask (booleana) deja (cuando la mask está en 255, True, y sino es porque está en False, 0)

"""

In [ ]:
# Join orange_masks and white_masks
fish_masks = [cv2.bitwise_or(o,w) for o, w in zip(orange_masks, white_masks)]

"""
el bit wise or, es una puerta lógica, que dice que si en alguna de las dos imagenes el pixel es naranja o blanco, da true 
Eso es porque comprueba las máscaras booleanas de ambos colores, y con que una dé True, ya da True el OR

"""

# Compute a list that contains the result of multiplying the original image with its complete mask.
fish = [cv2.bitwise_and(img, img, mask=mask) for img, mask in zip(imgs, fish_masks)]
# Aquí cogemos los pixeles donde la máscara de naranja o blanco haya dado True


In [ ]:
def load_images(filenames: List) -> List:
    return [imageio.imread(filename) for filename in filenames]


"""
Al cargar una imagen con **imageio.imread()**, los valores de los píxeles se almacenan en formato RGB. 

Sin embargo, cuando la mostramos con show_image() (que internamente utiliza OpenCV en **cv2.imshow()**), 
la librería interpreta las imágenes en formato BGR.

"""

# PRÁCTICA 2

---


In [ ]:
import cv2
print("OpenCV should be 4.8.0.76 Current version:", cv2.__version__)
from typing import List
import numpy as np
import imageio
import cv2
import copy
import glob # Librería necesaria para buscar nombres de paths con patrones concretos, usando *, etc.
import matplotlib.pyplot as plt

In [ ]:
# Contamos a mano cuantas esquinas de filas y columnas hay en el tablero de chess
pattern_size = (8,6)  
# Sigue el formato fila, columna. 
# Indica cuántas esquinas interiores hay que detectar en cada fila y cada columna 

"""
La función findChessboardCorners se encarga de encontrar las esquinas (corners) del tablero 
de cada una de las imágenes que se le pasen, devuelve una tupla de un booleano que indica si 
se ha encontrado el patrón completo del tablero o no, y luego un array que muestra las coordenadas 
de las esquinas detectadas. El primer parámetro de la función findChessboardCorners debe ser una 
imagen a color o escala de grises, por eso decidimos pasarla a escala de grises

"""
# Find corners with cv2.findChessboardCorners()
corners = [cv2.findChessboardCorners(
    cv2.cvtColor(img, cv2.COLOR_RGB2GRAY), pattern_size) 
    for img in imgs]  # Todo este proceso lo hacemos para cada imagen de las cargadas, por eso este bucle

# Es importante pasar como parámetro el número de esquinas interiores a detectar en el tablero

In [ ]:
# Hacemos por seguridad una copia de las esquinas detectadas
corners_copy = copy.deepcopy(corners)  

# Este es el criterio de parada que se usará en cornerSubPix, indica cuándo detener la iteración (por precisión alcanzada o por número máximo de iteraciones)
criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, 0.01) 
# En nuestro caso usaremos de condiciones tanto un máximo de 30 iteraciones, como precisión mínima a obtener de 0.01

# To refine corner detections with cv2.cornerSubPix() you need to input grayscale images. Build a list containing grayscale images.
imgs_gray = [cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) for img in imgs] # pasamos a grises las imágenes

# La función de cornerSubPix busca corners de manera más precisa ya que utiliza gradientes y consigue de forma iterativa encontrar la localización del corner de forma más exacta que findChessboardCorners
# A la función hay que pasarle como argumentos, la imagen del tablero, las coordenadas iniciales de las esquinas interiores que se habían detectado con findChessboardCorners, 
# el pattern_size de nuevo, el tamaño de una región sin usar en el centro de la ventana (se puede poner (-1, -1) para desactivar esta exclusión), y el criterio de parada del algoritmo
corners_refined = [cv2.cornerSubPix(i, cor[1], pattern_size, (-1, -1), criteria) if cor[0] else [] for i, cor in zip(imgs_gray, corners_copy)]

# A diferencia de findChessboardCorners, la función de cornerSubPix solo nos devuelve el array con las coordenadas encontradas de las esquinas inferiores. 
# Sin booleano de confirmación de que se encontró el patrón.

In [ ]:
# Use cv2.drawChessboardCorners() to draw the cornes

for i in range(len(corners_copy)):
    # Solo dibujamos las imágenes que tienen True en ret, es decir, que al buscar sus corners, se ha encontrado correctamente el patrón que se buscaba
    ret = corners_copy[i][0]
    if ret: # Este if es innecesario realmente porque al pasar luego ret como parámetro de drawChessboardCorners, se comprueba automáticamente, pero inicialmente no lo sabíamos y lo hicimos así
        # La función de drawChessboardCorners toma la imagen donde se va a dibujar el patrón de esquinas detectadas, el tamaño del patrón del tablero (esquinas interiores), 
        # las esquinas encontradas (le pasamos las refinadas que habíamos obtenido con cornerSubPix, y el booleano ret que hace que pintemos o no en función de si se encontró el patrón)
        cv2.drawChessboardCorners(imgs_copy[i], pattern_size, corners_refined[i], ret) # La función drawChessboardCorners no devuelve nada, solo modifica la imagen y dibuja sobre ella
        cv2.imshow(f"Img {i} and corners", imgs_copy[i])  # Una vez ya hemos sobreescrito la imagen, la mostramos. Se ve el tablero y las esquinas interiores encontradas sobre él
        cv2.waitKey(0)

cv2.destroyAllWindows()

In [ ]:
# Design the method. It should return a np.array with np.float32 elements
def get_chessboard_points(chessboard_shape, dx, dy):

    # to do comments
    cols, rows = chessboard_shape  # Se separan los valores de columnas y filas (número de esquinas interiores en cada dirección)

    grid_xy = np.mgrid[0:cols, 0:rows].T.reshape(-1,2) # Creamos una cuadrícula con las coordenadas 2D del tablero
    # np.mgrid genera una malla de coordenadas (una rejilla regular) con todos los pares (x, y) posibles
    # .T.reshape(-1,2) reorganiza la cuadrícula en una lista de puntos con forma (N, 2), donde N = cols * rows

    objp = np.zeros([grid_xy.shape[0], 3], dtype=np.float32)  # Creamos una matriz de ceros para los puntos 3D, que a continuación rellenaremos

    # dx y dy es el tamaño del espacio que hay entre esquinas en sentido vertical y horizontal
    objp[:,0] = grid_xy[:,0]*dx  # (coordenada X) se rellena multiplicando las posiciones de la cuadrícula por la distancia real entre esquinas en X (dx)
    objp[:,1] = grid_xy[:,1]*dy  # (coordenada Y) se rellena igual, pero con la distancia en Y (dy)

    # Devolvemos los puntos del tablero tal y como son en el mundo real, con su posición teórica exacta. 
    # Nos servirá para comparar esas posiciones con las posiciones encontradas por cornerSubPix y findChessboardCorners
    return objp  

In [ ]:
# Versión sencilla:

def get_chessboard_points(chessboard_shape, dx, dy):
    cols, rows = chessboard_shape
    obj_points = []

    # Recorremos filas y columnas como hicimos antes
    for y in range(rows):
        for x in range(cols):
            # Creamos el punto (X, Y, Z)
            # Multiplicamos la posición (0,1,2...) por la distancia real (dx, dy)
            # La Z siempre es 0 porque el tablero es plano
            punto_3d = [x * dx, y * dy, 0.0]
            obj_points.append(punto_3d)

    # El enunciado pide que devuelva un np.array de tipo float32
    return np.array(obj_points, dtype=np.float32)


"""
Crea un mapa de coordenadas del mundo real. 
Si tu tablero tiene cuadrados de 30 mm (dx=30,dy=30), 
esta función genera una lista que dice:

    Esquina 1: (0,0,0)

    Esquina 2: (30,0,0)

    Esquina 3: (60,0,0)

    ... y así sucesivamente.
"""

In [ ]:
# De esta forma obtenemos la ubicación (coordenadas x,y,z) de los pixeles que son esquinas de los tableros. Esta es la ubicación real de las esquinas, que será usada como object points más adelante
dx, dy = 30, 30
chessboard_points = [get_chessboard_points(pattern_size, dx, dy) for _ in corners_refined]

In [ ]:
img_size = (imgs_gray[0].shape[1], imgs_gray[0].shape[0]) 
valid_corners = [cor[1] for cor in corners if cor[0]]
valid_corners = np.asarray(valid_corners, dtype=np.float32)

"""
La función de calibrateCamera toma como argumentos los chessboard_points, 
que son los "objectPoints", es decir las coordenadas reales de las esquinas 
que hemos calculado nosotros usando  el grid y la distancia entre esquinas, 
luego los valid_corners, que son las posiciones de las esquinas que se han 
encontrado analizando las distintas imágenes usando findChessboardCorners,
y además se le pasa el tamaño de las imágenes (img_size) y otros 2 argumentos: 
InputOutputArray (cameraMatrix) y InputOutputArray (distCoeffs) que no usamos 
"""
rms, intrinsics, dist_coeffs, rvecs, tvecs = cv2.calibrateCamera(chessboard_points, valid_corners, img_size, None, None)
# calibrateCamera nos devuelve el error de reproyección (rms), los parámetros intrínsecos, y rvecs y tvecs que nos sirven para hallar los parámetros extrínsecos de la calibración de la cámara

# Obtain extrinsics
extrinsics = list(map(lambda rvec, tvec: np.hstack((cv2.Rodrigues(rvec)[0], tvec)), rvecs, tvecs))  # de esta forma obtenemos los parámetros extrínsecos de la calibración de la cámara

In [ ]:
# Parameters for cv2.fisheye.calibrate()

# Preparamos una serie de variables necesarias antes de llamar a cv2.fisheye.calibrate(), la función de OpenCV que realiza la calibración de una cámara con lente ojo de pez (fisheye camera model)
calibration_flags = cv2.fisheye.CALIB_RECOMPUTE_EXTRINSIC+cv2.fisheye.CALIB_FIX_SKEW  # Estas constantes controlan cómo se comporta el algoritmo de calibración, en este caso las sumamos
intrinsics = np.zeros((3, 3))  # Se inicializa como una matriz 3×3 de ceros que será rellenada por cv2.fisheye.calibrate() con los parámetros intrínsecos de la cámara
distortion = np.zeros((4, 1))  # Se inicializan como un vector columna 4×1 de ceros. El modelo fisheye de OpenCV utiliza cuatro parámetros de distorsión radial
rotations = [np.zeros((1, 1, 3), dtype=np.float64) for _ in imgs_corners]  # Se crea una lista donde cada elemento es un array 1×1×3 inicializado a cero (uno por cada imagen utilizada en la calibración)
traslations = [np.zeros((1, 1, 3), dtype=np.float64) for _ in imgs_corners]  # Igual que lo anterior, pero describe la posición del patrón respecto a la cámara


In [ ]:
# Esta línea ejecuta el proceso de calibración de cámara ojo de pez (fisheye), rellenando los parámetros que le habíamos definido antes
rms, _, _, _, _ = \
cv2.fisheye.calibrate(fisheye_chessboard_points,   # Lista de arrays con las coordenadas reales de las esquinas del tablero, expresadas en el sistema del mundo
                      imgs_corners, gray_img.shape[::-1],  # Lista de arrays con las coordenadas en píxeles de las esquinas detectadas en cada imagen, refinadas con cv2.cornerSubPix()
                      intrinsics, distortion, rotations, traslations, calibration_flags, # Los parámetros que queremos calibrar usando el algoritmo
                      subpix_criteria)  # Condición que define cuándo detener la optimización iterativa (por número máximo de iteraciones o por error mínimo alcanzado)

# La función estima los parámetros intrínsecos y de distorsión de la cámara de lente de pez, a partir de un conjunto de imágenes de un patrón conocido en nuestro caso un tablero de ajedrez

In [ ]:
# Search in the documentation to define 'dim'
dim = (fisheye_imgs[0].shape[0], fisheye_imgs[0].shape[1])  # pasamos las dimensiones de las imágenes que queremos corregir, que serán de (768, 1024)

# La función de fisheye.initUndistortRectifyMap no genera imágenes directamente, sino que calcula dos mapas de remapeo (map1 y map2) que contienen, para cada píxel de la imagen 
# rectificada (sin distorsión), la posición de dónde debe tomar su valor en la imagen original (distorsionada). Luego, esos mapas se usan en la función de cv2.remap
map1, map2 = cv2.fisheye.initUndistortRectifyMap(intrinsics, distortion, np.eye(3), intrinsics, dim, cv2.CV_16SC2)

In [ ]:
# Homework: correct distortion using cv2.remap()

# Hacemos una copia de seguridad por si acaso
undistorted_fisheye_imgs_copy = copy.deepcopy(fisheye_imgs) 

# Obtenemos las imágenes ya sin distorsión gracias a la función de remapm, que aplicamos para cada una de las imágenes iniciales
undistorted_fisheye_imgs = [cv2.remap(img, map1, map2, interpolation=cv2.INTER_LINEAR) for img in undistorted_fisheye_imgs_copy]
# cv2.remap() crea una nueva imagen tomando los valores de la imagen original y aplicandole el remapeo necesario mediante map1 y map2

# PRÁCTICA 3

---


### Parte A-B

In [ ]:
import numpy as np
import cv2
import os
import imageio
from typing import List
import glob
import matplotlib.pyplot as plt
import itertools

In [ ]:
def gaussian_blur(img: np.array, sigma: float, filter_shape: Optional[List] = None, verbose: bool = False) -> np.array:
    # If not given, compute the filter shape 

    # Para evitar aplicar ningun filtro gaussiano (útil para comparar imágenes con y sin ruido de apartados posteriores)
    if sigma == 0:
        if verbose:
            show_image(img=img, img_name=f"Gaussian Blur: Sigma = {sigma}")
        return None, img.astype(np.uint8)

    else:
        # Dependiendo de si el parámetro del tamaño del kernel se pasa como parámetro o no, sacamos las dimensiones del kernel de una o de otra forma
        if filter_shape == None:
            k = max(3, int(6*sigma + 1))  # en caso de no tener el tamaño del kernel, lo sacamos en función de la sigma del kernel
            if k % 2 == 0:
                k += 1  # Nos aseguramos de que sea un valor impar
            kh,kw = k,k

        else:  # En caso de sí disponer del tamaño del kernel, simplemente lo cogemos
            kh, kw = int(filter_shape[0]), int(filter_shape[1])
            if kh // 2 == 1: kh +=1
            if kw // 2 == 1: kw += 1  # Nos aseguramos de que sea un valor impar

        # TODO Create the filter coordinates matrices
        cy,cx = kh // 2, kw // 2
        y, x = np.mgrid[-cy:kh-cy, -cx:kw-cx]  # la función de np.mgrid genera dos arrays 2D con los índices de cada coordenada
        
        # TODO Define the formula that goberns the filter
        # formula = (1 / (2*np.pi*sigma**2))*np.exp(-(x**2+y**2)/(2*sigma**2))  # Definimos la fórmula de la Gaussiana pero aún no la hemos normalizado
        formula = np.exp(-(x**2+y**2)/(2*sigma**2))  # Definimos la fórmula de la Gaussiana pero aún no la hemos normalizado
        
        gaussian_filter = (formula / formula.sum()).astype(np.float32) # Aseguramos que la suma sea 1 (normalizando) para que la convolución no cambie el brillo global
        
        # TODO Process the image
        gb_img = cv2.filter2D(img, -1, gaussian_filter)  # Aplicamos el filtro gaussiano de blurreado a la imagen que se nos había pasado como parámetro
        
        if verbose:
            show_image(img=gb_img, img_name=f"Gaussian Blur: Sigma = {sigma}")
        
        # Devolvemos tanto el filtro gaussiano de blurreado que hemos generado, como la imagen a la que se lo hemos aplicado
        return gaussian_filter, gb_img.astype(np.uint8)

In [ ]:
# Version simplificada

def gaussian_blur_simple(img, sigma, filter_shape=None, verbose: bool = False):
    if sigma == 0:
        return None, img.astype(np.uint8)

    # 1. Determinar el tamaño del kernel (siempre impar)
    if filter_shape is None:
        k = max(3,int(6 * sigma + 1))
        if k % 2 == 0: k += 1
        kh = kw = k
    else:
        kh, kw = filter_shape
        # Asegurar que sean impares
        if kh % 2 == 0: kh += 1
        if kw % 2 == 0: kw += 1

    # 2. Crear el filtro vacío
    gaussian_filter = np.zeros((kh, kw), dtype=np.float32)
    cy, cx = kh // 2, kw // 2  # Centro del filtro

    # 3. Rellenar el filtro usando la fórmula (sustituye a mgrid)
    for i in range(kh):
        for j in range(kw):
            y = i - cy  # Distancia vertical al centro
            x = j - cx  # Distancia horizontal al centro
            
            # Aplicamos la fórmula de la Gaussiana
            exponente = -(x**2 + y**2) / (2 * sigma**2)
            gaussian_filter[i, j] = np.exp(exponente)

    # 4. Normalizar (para que la suma de todos los pesos sea 1)
    # Si no normalizamos, la imagen saldría mucho más brillante o blanca.
    gaussian_filter /= gaussian_filter.sum()

    # 5. Aplicar el filtro a la imagen
    # -1 significa que la imagen de salida tendrá la misma profundidad que la de entrada
    gb_img = cv2.filter2D(img, -1, gaussian_filter)

    return gaussian_filter, gb_img.astype(np.uint8)

In [ ]:
# Get the gaussian blurred images using a list comprehension
gauss_sigma = 1.4   
gb_imgs = [gaussian_blur(img,gauss_sigma,None,False)[1] for img in imgs]  # Aplicamos el filtro a cada imagen y guardamos el resultado

# Queremos blurrear para quitar el ruido y seguir procesando la imagen, 
# # ese es el objetivo de este primer paso con el filtro gaussiano

In [ ]:
def sobel_edge_detector(img: np.array, filter: np.array, gauss_sigma: float, gauss_filter_shape: Optional[List] = None, verbose: bool = False) -> np.array:
    # TODO Transform the img to grayscale
    gray_img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    
    # TODO Get a blurry img to improve edge detections
    blurred = gaussian_blur(img=gray_img, sigma=gauss_sigma, filter_shape=gauss_filter_shape, verbose=verbose)[1]
    # Accedemos al índice 1 ya que recordemos que la función de gaussian_blur devuelve el filtro usado y la imagen resultado
    
    # Re-scale
    blurred = blurred/255
    # Estandarizamos antes de calcular derivadas y ángulos ya que queremos trabajar siempre con magnitudes comparables y mantener esa consistencia
    
    # TODO Get vertical edges
    v_edges = cv2.filter2D(blurred, -1, filter)  # Usamos el filtro "filter" para obtener los bordes verticales
    
    # TODO Transform the filter to get the orthogonal edges
    filter = filter.T # para obtener el filtro de bordes horizontales, solo tenemos que trasponer el filtro que teníamos para los verticales
    
    # TODO Get horizontal edges
    h_edges = cv2.filter2D(blurred, -1, filter)
    
    # TODO Get edges
    # Calculamos la magintud del gradiente en cada punto, como la suma de los cuadrados del gradiente horizontal y del vertical
    sobel_edges_img = np.hypot(v_edges, h_edges)  # Equivalent to sqrt(x1**2 + x2**2)
    # Cuanto mayor sea el gradiente en ese punto, es que el cambio de intensidad es mayor y con más probabilidad sea un borde 
    
    # Get edges angle
    theta = np.arctan2(h_edges, v_edges) # Calculamos el ángulo usando la arcotangente de la parte vertical entre la horizontal
    # para posteriores etapas, nos es útil conocer la dirección/orientación en la que va el gradiente, es decir en la que cambia la intensidad    

    # Visualize if needed
    if verbose:
        show_image(img=sobel_edges_img, img_name="Sobel Edges")
    
    # La función de squeeze elimina posibles ejes de longitud 1 que se nos hayan quedado por ahí al hacer operaciones con los arrays
    return np.squeeze(sobel_edges_img), np.squeeze(theta)

In [ ]:
gauss_sigma = 1.4

# Define the Sobel filter
sobel_filter = np.array([[-1, 0, 1],
                         [-2, 0, 2],
                         [-1, 0, 1]], dtype=np.float32)   # Usamos el kernel indicado en el enunciado (es el que se usa para detectar bordes verticales)


# TODO Get the edges detected by Sobel using a list comprehension
sobel_edges_imgs = [sobel_edge_detector(img=img, filter=sobel_filter, gauss_sigma=gauss_sigma, gauss_filter_shape=None, verbose=False)[0]
                    for img in imgs]

In [ ]:
def canny_edge_detector(img: np.array, sobel_filter: np.array, gauss_sigma: float, gauss_filter_shape: Optional[List] = None, verbose: bool = False):
    # TODO Call the method sobel_edge_detector()
    sobel_edges_img, theta = sobel_edge_detector(img=img, filter=sobel_filter, gauss_sigma=gauss_sigma, gauss_filter_shape=gauss_filter_shape, verbose=verbose)
    
    # TODO Use NMS to refine edges
    canny_edges_img = non_max_suppression(img=sobel_edges_img, theta=theta)  
    # La función de non max supression (utils.py) nos proporciona una imagen que ya tiene los bordes detectados, pero aún más refinados
        
    return canny_edges_img

In [ ]:
# Nos creamos una función que automáticamente añade ruido gaussiano a las imágenes
def add_gaussian_noise(img, mean=0, sigma=25):
    noisy = img + np.random.normal(mean, sigma, img.shape)
    noisy = np.clip(noisy, 0, 255).astype(np.uint8)  # Aseguramos que los valores se queden en un rango válido
    return noisy

# Probamos a aplicar la función a una de nuestras imágenes
noisy_gauss = add_gaussian_noise(imgs[0], sigma=16)
show_image(noisy_gauss)


In [ ]:
# añadir ruido manualmente
import random

filas, columnas, canales = imgs[0].shape
img_gray = imgs # [cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) for img in imgs]

for f in range(filas):
    for c in range(columnas):
        for ch in range(canales):

            if img_gray[0][f][c][ch] < 255-50:
                img_gray[0][f][c][ch] = img_gray[0][f][c][ch] + random.randint(0,20)

            elif img_gray[0][f][c][ch] > 50:
                img_gray[0][f][c][ch] = img_gray[0][f][c][ch] - random.randint(0,50)

cv2.imshow("foto tras ruido", img_gray[0])
cv2.waitKey(0)
cv2.destroyAllWindows()


In [ ]:
# Scikit Image

# Comparación filtros Sobel, Prewitt y Canny
from skimage import io, color, filters, feature  

for i, img in enumerate(imgs):
    # Convertimos a escala de grises para facilitar la detección de bordes
    gray = color.rgb2gray(img)

    # Aplicamos los tres detectores que nos pide el enunciado
    edges_sobel = filters.sobel(gray)      # Gradiente con máscara Sobel
    edges_prewitt = filters.prewitt(gray)  # Gradiente con máscara Prewitt
    edges_canny = feature.canny(gray, sigma=1.5)  # Detector Canny con suavizado interno

    # # Mostramos resultados
    # fig, ax = plt.subplots(1, 4, figsize=(16, 6))
    # ax[0].imshow(gray, cmap='gray'); ax[0].set_title("Original (gris)"); ax[0].axis("off")
    # ax[1].imshow(edges_sobel, cmap='gray'); ax[1].set_title("Sobel")
    # ax[2].imshow(edges_prewitt, cmap='gray'); ax[2].set_title("Prewitt")
    # ax[3].imshow(edges_canny, cmap='gray'); ax[3].set_title("Canny")
    # plt.suptitle(f"Comparación de detectores en imagen {i}")
    # plt.show()


In [ ]:
def binarize(img: np.array, threshold: int = 127):
    # Convertimos a escala de grises si la imagen es RGB
    if len(img.shape)== 3:
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    else:
        gray = img

    # Aplicamos umbralización manualmente usando numpy
    binary_img = np.where(gray < threshold, 0, 255).astype(np.uint8)
    # Todo lo que es menor que el threshold, lo ponemos como 0, y el resto como 1

    # Aplicamos umbralización usando threshold
    # _, binary_img = cv2.threshold(gray, threshold, 255, cv2.THRESH_BINARY)
    # Todo lo que es menor que el threshold, se pone como 0, y el resto como 1

    return binary_img

show_image(imgs[0])
show_image(binarize(imgs[0]))


In [ ]:
# Define the dilation method

# Definimos el tamaño de nuestro kernel, que actuará como elemento estructurante para el operador morfológico de dilatación
kernel_shape = (3,3)
structurant_element = np.ones(shape=kernel_shape).astype(np.uint8)

def custom_dilate(img):

    # Binarizamos la imagen antes de nada (por si acaso nolo está)
    img = binarize(img)

    # TODO pad the original image so it can keep dimensions after processing
    pad_width = int((kernel_shape[0] - 1) / 2)  # investigando en internet sobre cómo se relaciona el tamaño del kernel aplicado y el padding necesario, hallamos esta fórmula

    padded = np.pad(array=img, pad_width=pad_width, mode="mean")  # hacemos padding usando un kernel de (3x3) por lo que tenemos que añadir solo 1 fila y columna tanto arriba como abajo de la imagen
    # si usamos el modo "mean" significa que el valor que se añade de padding será la media de los valores cercanos en ese pixel de la imagen
    
    # TODO get img shape
    width = img.shape[1]
    height = img.shape[0]
    
    # TODO Create an element with the same dimensions as the padded img
    dilated = np.zeros(shape=padded.shape)

    # Preferimos hacer el bucle usando los indices con la referencia ya en la matriz con padding añadido, para verlo más claro
    for j in range(pad_width, pad_width + height):
        for i in range(pad_width, pad_width + width):

            # TODO Add logic to the operation

            # Primero seleccionamos cuál es el área que tenemos que tener en cuenta para esta iteración según nuestro elemento estructurante
            area_involved = padded[j - pad_width:j + pad_width + 1,
                                i - pad_width:i + pad_width + 1]*structurant_element
            # cogemos una matriz de tamaño kernel_shape, centrada en nuestro elemento j,i y luego lo multiplicamos por el elemento estructurante
            
            # Cogemos el máximo de todos los valores de nuestro área seleccionada, ese será el nuevo valor asociado al pixel j,i (dilatado)
            dilated[j, i] = np.max(area_involved)

    # TODO Select the region of interest (ROI). Modify if needed
    dilated = dilated[pad_width:height+1, pad_width:width+1]  # a la hora de devolver la imagen, nos quedamos con lo que no es parte añadida por el padding
    
    return dilated.astype(np.uint8)  # devolvemos la imagen convertida a una matriz de números enteros

In [3]:
# Ejemplo para entender mejor como funciona la función de np.pad()
import numpy as np
np.pad(np.array([[1,2,3], [4,5,6]]), pad_width=1, constant_values="0")

array([[0, 0, 0, 0, 0],
       [0, 1, 2, 3, 0],
       [0, 4, 5, 6, 0],
       [0, 0, 0, 0, 0]])

In [ ]:
# Define the erotion method

# Definimos el tamaño de nuestro kernel, que actuará como elemento estructurante para el operador morfológico de dilatación
kernel_shape = (3,3)
structurant_element = np.ones(shape=kernel_shape).astype(np.uint8)

def custom_erode(img):

    # Binarizamos la imagen antes de nada (por si acaso nolo está)
    img = binarize(img)

    # TODO pad the original image so it can keep dimensions after processing
    pad_width = int((kernel_shape[0] - 1) / 2)  # investigando en internet sobre cómo se relaciona el tamaño del kernel aplicado y el padding necesario, hallamos esta fórmula

    padded = np.pad(array=img, pad_width=pad_width, mode="mean")  # hacemos padding usando un kernel de (3x3) por lo que tenemos que añadir solo 1 fila y columna tanto arriba como abajo de la imagen
    # si usamos el modo "mean" significa que el valor que se añade de padding será la media de los valores cercanos en ese pixel de la imagen
    
    # TODO get img shape
    width = img.shape[1]
    height = img.shape[0]
    
    # TODO Create an element with the same dimensions as the padded img
    eroded = np.zeros(shape=padded.shape)

    # Preferimos hacer el bucle usando los indices con la referencia ya en la matriz con padding añadido, para verlo más claro
    for j in range(pad_width, pad_width + height):
        for i in range(pad_width, pad_width + width):

            # TODO Add logic to the operation

            # Primero seleccionamos cuál es el área que tenemos que tener en cuenta para esta iteración según nuestro elemento estructurante
            area_involved = padded[j - pad_width:j + pad_width + 1,
                                i - pad_width:i + pad_width + 1]*structurant_element
            # cogemos una matriz de tamaño kernel_shape, centrada en nuestro elemento j,i y luego lo multiplicamos por el elemento estructurante
            
            # Cogemos el máximo de todos los valores de nuestro área seleccionada, ese será el nuevo valor asociado al pixel j,i (dilatado)
            eroded[j, i] = np.min(area_involved)

    # TODO Select the region of interest (ROI). Modify if needed
    eroded = eroded[pad_width:height+1, pad_width:width+1]  # a la hora de devolver la imagen, nos quedamos con lo que no es parte añadida por el padding
    
    return eroded.astype(np.uint8)  # devolvemos la imagen convertida a una matriz de números enteros


In [ ]:
# Imágenes dilatadas
dilated_imgs = [custom_dilate(img) for img in imgs]

# Imágenes erosionadas
eroded_imgs = [custom_erode(img) for img in imgs]

### Parte C

In [ ]:
import numpy as np
import cv2
import os
import imageio
from typing import List
import glob
import matplotlib.pyplot as plt
import itertools

In [ ]:
# Create a folder to save all partC results (inside data)
folder_name = "partC_processed"  # el nombre que va a tener la carpeta
folder_path = os.path.join("..", "data", "partC", "source", folder_name)  
# como tiene que estar dentro de la carpeta data, tenemos que salir de la carpeta src en la que estamos actualmente ".."
os.makedirs(name=folder_path, exist_ok=True)  # usamos exist_ok = True por si acaso la carpeta ya existe para que no salte ningún error o excepción

In [ ]:
# Define Harris Corner detection function
def harris_corner_detector(image: np.array, blockSize: int, ksize: int, k: float):
    '''
    image - Input image 
    blockSize - Size of neighborhood considered for corner detection
    ksize - Aperture parameter of the Sobel derivative used
    k - Harris detector free parameter in the equation.
    '''
    # TODO Input image to Harris corner detector should be grayscale 
    gray = cv2.cvtColor(src=image, code=cv2.COLOR_BGR2GRAY)

    # TODO Input image to Harris corner detector should be float32 type
    gray = gray.astype(np.float32)

    # TODO Apply Harris corner detection
    harris = cv2.cornerHarris(src=gray, blockSize=blockSize, ksize=ksize, k=k)
    
    # Result is dilated for marking the corners, not important
    harris = cv2.dilate(src=harris, kernel=None)
    
    # TODO Threshold for an optimal value of 1% of maximal R value
    # If pixel value > 1% max value, yo should to hightlight this as a red corner
    max_val = np.max(harris)  # sacamos el valor máximo
    th = 0.01*max_val  # definimos nuestro usando 0.01 
    for fila in range(harris.shape[0]):
        for columna in range(harris.shape[1]):
            if harris[fila][columna] > th:  # comparamos todos los pixeles con el valor del threshold
                image[fila][columna] = [0, 0, 255]  # en caso de complirse la condición, ponemos un máximo de rojo (esta en BGR por lo que es el tercer canal)
                # el marcado en rojo se hace inplace sobre image, por eso convendrá hacer una copia de dicha imagen antes de usar esta función

    return image

In [7]:
a = np.array([1,2,3])
print(a.dtype)
a = np.array(a, np.float32)
print(a.dtype)


int32
float32


In [ ]:
# Define Shi-Tomasi corner detection function
def shi_tomasi_corner_detection(image: np.array, maxCorners: int, qualityLevel:float, minDistance: int, corner_color: tuple, radius: int):
    '''
    image - Input image
    maxCorners - Maximum number of corners to return. 
                 If there are more corners than are found, the strongest of them is returned. 
                 maxCorners <= 0 implies that no limit on the maximum is set and all detected corners are returned
    qualityLevel - Parameter characterizing the minimal accepted quality of image corners. 
                   The parameter value is multiplied by the best corner quality measure, which is the minimal eigenvalue or the Harris function response. 
                   The corners with the quality measure less than the product are rejected. 
                   For example, if the best corner has the quality measure = 1500, and the qualityLevel=0.01 , then all the corners with the quality measure less than 15 are rejected
    minDistance - Minimum possible Euclidean distance between the returned corners
    corner_color - Desired color to highlight corners in the original image
    radius - Desired radius (pixels) of the circle
    '''
    # TODO Input image to Tomasi corner detector should be grayscale 
    gray = cv2.cvtColor(src=image, code=cv2.COLOR_BGR2GRAY)

    # TODO Apply cv2.goodFeaturesToTrack function
    corners = cv2.goodFeaturesToTrack(
        image=gray, 
        maxCorners=maxCorners, 
        qualityLevel=qualityLevel, 
        minDistance=minDistance)  # obtenemos los corners llamando a la función de goodFeaturesToTrack que hereda los parámetros que ya nos han pasado previamente
    
    # TODO Corner coordinates conversion to integers
    corners = np.intp(corners)  # con esta función de numpy conseguimos que todas las coordenadas sean del tipo entero
    
    for corner in corners:  # iteramos por cada uno de los corners encontrados, y hacemos un círculo sobre él para marcarlo bien

        # Multidimensional array into flattened array, if necessary
        x, y = corner.ravel()  # de esta forma obtenemos las coordenadas de cada uno de los corners detectados, que nos sirven como centro del círculo que vamos a trazar

        # TODO Circle corners
        cv2.circle(image, center=(x,y), radius=radius, color=corner_color)  
        # el círculo se hace inplace sobre image, por eso convendrá hacer una copia de dicha imagen antes de usar esta función

    return image

# PRÁCTICA 4

---


### Parte A

In [ ]:
import cv2
import os
import matplotlib.pyplot as plt
from rpds import List
import numpy as np

In [ ]:
# Apply canny to the images, play with the thresholds to get a good result

edge_imgs = []  # to store the canny edge images for later Hough transform

# 1st image

# Hyperparameters (para la histéresis)
low_threshold = 30
high_threshold = 180

# get the first image
img = imgs[0]

# Convert to grayscale
gray_img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

# Apply Canny edge detection
edge_img = cv2.Canny(gray_img, low_threshold, high_threshold)

# Store the edge image for later Hough transform
edge_imgs.append(edge_img)

# Display the image or save it as you like
show_image(edge_img, "Canny Edge Image 1")

In [ ]:
# Draw the lines on the images. A line is a tuple of 4 integers (x1, y1, x2, y2) where (x1, y1) and (x2, y2) are the coordinates of the two points of the line.
def draw_lines(img, lines, color, thickness):

    # Iteramos por cada una de las líneas detectadas
    for line in lines:
        # Extraemos las coordenadas x,y de los 2 puntos que definen la línea
        x1, y1, x2, y2 = line[0]
        # Pintamos la línea sobre la imnagen, sobreescribiéndola
        cv2.line(img, (x1, y1), (x2, y2), color, thickness)

In [ ]:
# Apply the hough transform to the images, play with the parameters to get a good result

# Hacemos una copia de las imágenes originales ya que al pintar sobre ellas las estaremos sobreescribiendo
new_imgs1 = load_images(imgs_path)

rho = 1
theta = np.pi/180
threshold = 1
min_line_length = 10
max_line_gap = 10

# Get the edge image
edges1 = edge_imgs[0]

show_image(new_imgs1[0])

# Apply the hough transform
lines = cv2.HoughLinesP(edges1, rho, theta, threshold, minLineLength=min_line_length, maxLineGap=max_line_gap)

# Draw the lines on the image (on the original image, make sure to get a copy, as draw_lines modifies the image)
draw_lines(new_imgs1[0], lines, (0, 255, 0), 2)

# Show the image or save it as you like
show_image(new_imgs1[0], "Hough Lines Image 1")

### Parte B

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from utils import *
from bow import BoW
from dataset import Dataset
from image_classifier import ImageClassifier
import time
from tqdm import tqdm
import sys
import pickle
import os
from rpds import List


In [ ]:
def generateGaussianImages(image, sigmas):
    """Generate the gaussian images using the base image and the sigmas given

    Args:
        image (np.array[np.float32]): Base image to blur
        sigmas (List[np.float32]): Sigmas for blurring the image 

    Returns:
        List[np.array[np.float32]: List of blurred images
    """
    gaussian_images = []

    # TODO: Generate the list of blurred images using cv2.GaussianBlur()
    for sigma in sigmas:
        
        k = max(3, 2*round(3*sigma + 1))  # en caso de no tener el tamaño del kernel, lo sacamos en función de la sigma del kernel
        if k % 2 == 0:
            k += 1  # Nos aseguramos de que sea un valor impar
    
        blurred_image = cv2.GaussianBlur(image, (k, k), sigmaX=float(sigma), sigmaY=float(sigma))
        gaussian_images.append(blurred_image)
    
    return gaussian_images
    

In [ ]:
# Load the image in grayscale

# Cargamos la imagen original (la no rotada) y la convertimos a escala de grises
image = cv2.imread(imgs_path[0], cv2.IMREAD_GRAYSCALE)
image = image.astype(np.float32)  # Convertimos la imagen a float32

# Number of diference of gaussians to generate (minimum 3)
# TODO: Adjust this number to make sure keypoints are generated
intervals = 5  # Elegimos usar 5+1=6 gaussianas distintas para detectar los máximos y mínimos 

# Initial sigma
sigma = 1.6  # Sigma inicial de las 6 que va a haber en total

# We get the necessary sigmas from the given method generateGaussianSigmas()

# Función perteneciente a utils.py que nos permite generar una lista de sigmas, kernels gaussianos para aplicar el blur a la imagen
sigmas = sorted(generateGaussianSigmas(sigma, intervals))  #  recibe la sigma inicial y el número de diferencias de gaussianas que queremos generar

#TODO: Generate the gaussian images
gaussian_images = generateGaussianImages(image, sigmas)


In [ ]:
def generateDoGImages(gaussian_images):
    """

    Generate Difference-of-Gaussians list

    Args:
        gaussian_images (List[np.array[np.float32]): List of blurred images

    Returns:
        List[np.array[np.float32]: List of difference of gaussian images

    """

    dog_images = []  # Inicializamos la lista vacía donde vamos a almacenar las imágenes resultado

    # TODO: Generate the list of difference of gaussians using cv2.subtract()
    for i in range(len(gaussian_images) - 1):  # Iteramos desde la primera foto hasta la penúltima, sin llegar a la última para no salirnos del rango al operar la resta
        dog_img = cv2.subtract(gaussian_images[i+1], gaussian_images[i]) # Aplicamos la operación: Imagen i+1 ésima - Imagen i ésima
        dog_images.append(dog_img.astype(np.float32))  # Guardamos el resultado en nuestra lista
    
    return dog_images

dog_images = generateDoGImages(gaussian_images)

In [9]:
abs(np.array([-1,2]))

array([1, 2])

In [12]:
a = np.array([[1,2], [2,2]])
print(a)
print()
print(np.array(a))

[[1 2]
 [2 2]]

[[1 2]
 [2 2]]


In [ ]:
def isPixelAnExtremum(first_subimage, second_subimage, third_subimage, threshold):
    """Return True if the center element of the 3x3x3 array composed of subimages is strictly greater than or less than all its neighbors, False otherwise

    Args:
        first_subimage (np.array): Patch from first gaussian
        second_subimage (np.array): Patch from second gaussian
        third_subimage (np.array): Patch from third gaussian
        threshold (float): Value threshold for the pixel

    Returns:
        Bool: True if maximum or minimum, False otherwise
    """

    # TODO: Check if the center pixel value is a maximum or a minimum and if its absolute value is higher than the threshold
    center_value = float(second_subimage[1, 1])

    if abs(center_value) < float(threshold):  # Si el valor es inferior en valor absoluto al threshold, directamente devolvemos False
        return False
    
    # La segunda comprobación es la de que sea el máximo entre sus vecinos 
    neighbor_values = []
    for plane in (first_subimage, second_subimage, third_subimage):
        for i in range(3):  # Porque usamos estructura de 3x3
            for j in range(3):  # Porque usamos estructura de 3x3
                if i == 1 and j == 1 and plane is second_subimage: # no añadimos el propio pixel central como vecino
                    continue  # Skip the center pixel itself 
                neighbor_values.append(float(plane[i, j]))  # añadimos los vecinos a nuestro pixel, de cada una de las capas (primera, segunda y tercera)
                    
    # Comprobamos si el valor de nuestro pixel, o es mayor que todos, o es menor que todos los vecinos. En caso de ser así: True, sino: False
    extremum = (center_value > max(neighbor_values)) or (center_value < min(neighbor_values))

    return extremum  # Devolvemos el booleano como resultado

In [ ]:
# Hacemos una prueba para comprobar que la función de isPixelAnExtremum está bien implementada

# Creamos 3 capas enteras de ceros, pero poniendo el elemento central (de la segunda capa) como un 1
first_subimage = np.zeros((3,3),np.float32)
third_subimage = np.zeros((3,3),np.float32)
second_subimage = np.zeros((3,3),np.float32)
second_subimage[1,1] = 1
threshold = 0.03

isPixelAnExtremum(first_subimage,second_subimage, third_subimage, threshold)  # Debe dar True

In [ ]:
def findScaleSpaceExtrema(gaussian_images, dog_images, num_intervals, sigma, threshold=0.03):
    """Find pixel positions of all scale-space extrema in the image pyramid """
    keypoints = []  # Inicializamos una lista de puntos clave que vamos a ir rellenando

    # TODO: Fill the loop source data
    for image_index in range(len(dog_images) - 2):  # Cogemos trios desde las imágenes (0,1,2) hasta las (n-2, n-1, n)
        first_image = dog_images[image_index]  # Primera capa (primer índice)
        second_image = dog_images[image_index + 1]  # Segunda capa (segundo índice)
        third_image = dog_images[image_index + 2]  # Tercera capa (tercer índice)

        # (i, j) is the center of the 3x3 array

        # TODO: Fill the 2 range limits knowing you have to move the 3x3 window across the whole image
        for i in range(1,second_image.shape[0]-1):  # Nos tenemos que saltar el índice 0 porque al usar patch de 3x3 nos saldríamos del rango
            for j in range(1,second_image.shape[1]-1):  # Nos tenemos que saltar el índice 0 porque al usar patch de 3x3 nos saldríamos del rango
                # TODO: Fill the method with the required arguments
                if isPixelAnExtremum(first_image[i-1:i+2, j-1:j+2], # Cogemos desde la fila anterior hasta la fila siguiente y lo mismo con columnas (patch 3x3)
                                    second_image[i-1:i+2, j-1:j+2], 
                                    third_image[i-1:i+2, j-1:j+2],
                                    threshold):  # Usamos el threshold que nos hayan pasado como argumento
                    
                    # Refine the keypoint localization
                    localization_result = localizeExtremumViaQuadraticFit(i, j, image_index + 1, num_intervals, dog_images, sigma)

                    """
                    localizeExtremumViaQuadraticFit toma la posición inicial (i,j)
                    y la refina buscando el verdadero máximo/minimo de la función DoG
                    mediante una aproximación en Taylor de segundo orden en 3D.

                    """

                    if localization_result is not None:
                        keypoint, localized_image_index = localization_result  # solo si el refinamiento ha sido satisfactorio
                        # Get the keypoint orientation
                        keypoints_with_orientations = computeKeypointsWithOrientations(keypoint, gaussian_images[localized_image_index])

                        """
                        computeKeypointsWithOrientations calcula el gradiente
                        alrededor del punto clave en la imagen gaussiana y asigna
                        la orientación principal y secundarias detectadas

                        """
                        for keypoint_with_orientation in keypoints_with_orientations:
                            # Finalmente añadimos todos los keypoints generados (puede haber más de uno)
                            keypoints.append(keypoint_with_orientation)
    return keypoints

# Keypoint Detection with Taylor Expansion and Contrast Thresholding
# Tras múltiples pruebas hemos optado por usar este valor de threshold. Más adelante aportamos un análisis más detallado del significado de este hiperparámetro
threshold = 0.1 
keypoints = findScaleSpaceExtrema(gaussian_images, dog_images, intervals, sigma, threshold)
print(f"Keypoints - Threshold = {threshold}")
visualizeKp(image.copy(), keypoints)  # Llamamos a la función usando una copia de la imagen original, por si acaso, no queremos sobreescribirla
keypoints = removeDuplicateKeypoints(keypoints)


In [ ]:
# Utilizamos la función que se nos proporciona en el fichero utils.py
descriptors = generateDescriptors(keypoints, gaussian_images)

"""
La función generateDescriptors toma los keypoints detectados (con posición, orientación y escala 
refinadas), las imágenes gaussianas (pirámide), y devuelve un descriptor de 128 dimensiones por 
cada keypoint. Este es el descriptor de SIFT

"""

In [ ]:
def computeKeypointsAndDescriptors(image, sigma=1.6, num_intervals=5, th=0.8):
    """Compute SIFT keypoints and descriptors for an input image """
    # TODO: Fill the pipeline to get the keypoint and descriptors as before
    
    # Asegurarnos de que trabajamos con float32, como en SIFT original
    image = image.astype(np.float32)

    # Generamos los kernels gaussianos (las sigmas correspondientes)
    gaussian_kernels = sorted(generateGaussianSigmas(sigma, num_intervals))

    # Generamos la pirámide de imágenes gaussianas
    gaussian_images = generateGaussianImages(image, gaussian_kernels)

    # Generamos las diferencias de gaussianas (DoG)
    dog_images = generateDoGImages(gaussian_images)

    # Detectamos los keypoints en el espacio de escalas
    keypoints = findScaleSpaceExtrema(gaussian_images, dog_images, num_intervals, sigma, threshold=th)
    keypoints = removeDuplicateKeypoints(keypoints)

    # Generamos los descriptores para los keypoints
    descriptors = generateDescriptors(keypoints, gaussian_images)

    return keypoints, descriptors

In [ ]:
# TODO: Load the images with cv2 in grayscale

img1 = cv2.imread(imgs_path[0], cv2.IMREAD_GRAYSCALE)
img2 = cv2.imread(imgs_path[1], cv2.IMREAD_GRAYSCALE)

# TODO: Fill the function calls
# Obtenemos tanto los keypoints como los descriptores de ambas imágenes (la original y la rotada)
kp1, des1 = computeKeypointsAndDescriptors(img1, th=4.8)
kp2, des2 = computeKeypointsAndDescriptors(img2, th=4.8)

print("Resultados de la extracción de keypoints y descriptores:")
print("KP1 =", len(kp1), "- DES1 =", None if des1 is None else des1.shape)
print("KP2 =", len(kp2), "- DES2 =", None if des2 is None else des2.shape)

visualizeKp(img1.copy(), kp1)
visualizeKp(img2.copy(), kp2)

# Establecemos la correspondencia entre los puntos de interés de ambas imágenes
matchFeatures(img1, kp1, des1, img2, kp2, des2) # los des son descriptores


In [ ]:
# Shift OpenCV

# Create a SIFT detector object
sift = cv2.SIFT_create()

# Detect keypoints and compute descriptors
keypoints_0, descriptors_0 = sift.detectAndCompute(image_0, None)
keypoints_1, descriptors_1 = sift.detectAndCompute(image_1, None)

# Draw keypoints on the image
image_with_keypoints_0 = cv2.drawKeypoints(image_0, keypoints_0, None, flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS)
image_with_keypoints_1 = cv2.drawKeypoints(image_1, keypoints_1, None, flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS)

plt.imshow(image_with_keypoints_0)
plt.title('Keypoints with Blob Sizes and Orientations')
plt.axis('off')
plt.show()
plt.imshow(image_with_keypoints_1)
plt.title('Keypoints with Blob Sizes and Orientations')
plt.axis('off')
plt.show()

# Initialize BFMatcher
bf = cv2.BFMatcher(cv2.NORM_L2, crossCheck=False)

# Perform KNN matching with k=2 for ratio test
matches = bf.match(descriptors_0, descriptors_1)

# sort matches by distance
matches = sorted(matches, key = lambda x:x.distance)


# Draw the good matches
matched_image = cv2.drawMatches(image_0, keypoints_0, image_1, keypoints_1, matches[:50], None,
                                flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS)

# Display the matched features
plt.figure(figsize=(12, 6))
plt.imshow(matched_image)
plt.title("Feature Matches")
plt.axis('off')
plt.show()

In [ ]:
image_1 = cv2.rotate(image_0, cv2.ROTATE_90_CLOCKWISE)

In [ ]:
# Cargamos los datasets usando la extensión de jpg que tienen todas las imágenes que están en dicho dataset
training_set = Dataset.load("../data/partB/dataset/training", file_extension="jpg")
validation_set = Dataset.load("../data/partB/dataset/validation", file_extension="jpg")

In [ ]:
feature_extractor = cv2.SIFT_create()

"""
crea un objeto SIFT oficial de OpenCV, que tiene dos funciones clave:

detect() detecta keypoints
compute() calcula descriptores SIFT
detectAndCompute() hace ambos a la vez

"""

# Extract features
print("\nComputing SIFT descriptors...")
time.sleep(0.1)  # Prevents a race condition between tqdm and print statements.
descriptors = []

for path in tqdm(training_set, unit="image", file=sys.stdout):
    # TODO: Load the image from the path in grayscale
    image = cv2.imread(path, cv2.IMREAD_GRAYSCALE)  # importante añadir la flag para que se cargue la imagen en escala de grises
    try:
        # TODO: Using the cv2 methods get the descriptors for the image
        _, descriptor = feature_extractor.detectAndCompute(image, None)  # Usamos el método del feature_extractor que habíamos creado
        # Aquí podríamos llamar también a nuestra función de computeKeypointsAndDescriptors, pero se nos pide usar métodos de cv2
    except:
        print(f"WARN: Issue generating descriptor for image {path}")

    if descriptor is not None:
        descriptors.append(descriptor)

In [ ]:
"""

Se va a entrenar un clasificador que sea capaz de diferenciar entre todos estos grupos de palabras visuales. 

Este será un modelo basado en una Support Vector Machine(SVM). 

Con este clasificador podremos recibir imágenes externas al dataset de entrenamiento y clasificarlas en nuestro grupo de palabras.

"""

bow = BoW()
# TODO: Especify the args for the loading method
bow.load_vocabulary(filename="vocabulary")  
# le pasamos el nombre del fichero .pickle (sin la extensión que si no da error) donde habíamos guardado nuestro vocabulario de palabras visuales

image_classifier = ImageClassifier(bow)
# TODO: Especify the args for the training method
image_classifier.train(training_set=training_set)  # indicamos cuál tiene que ser el dataset de entrenamiento para el modelo
classifier = "classifier"  
image_classifier.save(classifier)  # guardamos el modelo ya entrenado en un fichero .xml y con las etiquetas de categoría en un .json 

In [ ]:
bow = BoW()

# TODO: Especify the args for the loading method
bow.load_vocabulary(filename="vocabulary")

image_classifier = ImageClassifier(bow)
# TODO: Especify the args for the loading method
image_classifier.load(filename=classifier)

# TODO: Especify the args for the loading method
image_classifier.predict(dataset=training_set, dataset_name="training_set")  # Este método automáticamente genera la matriz de confusión y nos proporciona el accuracy (0.58 en este caso)
# También nos genera un excel de resultados (training_set_results.xlsx) donde podemos analizar los resultados mejor

In [ ]:
bow = BoW()
# TODO: Especify the args for the loading method
bow.load_vocabulary(filename="vocabulary")

image_classifier = ImageClassifier(bow)

# TODO: Especify the args for the loading method
image_classifier.load(classifier)

# TODO: Especify the args for the loading method
image_classifier.predict(dataset=validation_set, dataset_name="validation_set")

### Parte C

In [ ]:
import cv2
import os
import numpy as np
import time

In [ ]:
#TODO: Use VideoCapture from OpenCV to read the video (visiontraffic.avi)
videopath = "../data/partC/visiontraffic.avi"  # Path to the video file
cap = cv2.VideoCapture(filename=videopath)  # Complete this line to read the video file

#TODO: Check if the video was successfully opened
if not cap.isOpened():  # Comprobamos que se abrió bien el video
    print('Error: Could not open the video file')
    os._exit(0)

#TODO: Get the szie of frames and the frame rate of the video
frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
frame_rate = cap.get(cv2.CAP_PROP_FPS)

# Añadimos un print para mostrar las características del video con el que estamos trabajando
print("Video properties:")
print(" - Width :", frame_width)
print(" - Height:", frame_height)
print(" - FPS   :", frame_rate)

#TODO: Use a loop to read the frames of the video and store them in a list
frames = []
while True:
    ret, frame = cap.read() # Esto me permite leer cada uno de los frames que componen el video (al final el vídeo es una sucesión de imágenes)
    if not ret:
        break
    frames.append(frame)

cap.release()

print(f"\nTotal frames read: {len(frames)}")

In [ ]:
# Para ver el video cargado
cap = cv2.VideoCapture(videopath)

delay = int(1000 / frame_rate)   # delay en ms

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break
    
    cv2.imshow("Video Cargado", frame)
    if cv2.waitKey(delay) & 0xFF == ord('q'):  # Pulsar 'q' para salir
        break

cap.release()
cv2.destroyAllWindows()

In [ ]:
#TODO:  Show the frames to select the reference frame, press 'n' to move to the next frame and 's' to select the frame
for i, frame in enumerate(frames):
    # Show the frame
    cv2.imshow(f'Video frame {i}', frame)
    # Wait for the key
    key = cv2.waitKey(0)
    # If the key is 'n' continue to the next frame
    if key == ord('n'):
        continue
    # If the key is 's' select the frame as the reference frame
    elif key == ord('s'):
        # Copy the frame to use it as a reference
        reference_frame = frame
        # Convert the reference frame to grayscale
        reference_frame = cv2.cvtColor(reference_frame, cv2.COLOR_BGR2GRAY)
        print('Frame {} selected as reference frame'.format(i))
        break

cv2.destroyAllWindows()

In [ ]:
#TODO: Compute the difference between the reference frame and the rest of the frames and show the difference
for frame in frames:
    # time.sleep(0.001)  # Este sleep es por si queremos ver el vídeo más lentamente y fijarnos bien 
    # Convert the frame to grayscale
    frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    diff = cv2.absdiff(reference_frame, frame)
    cv2.imshow('Diferencia', diff)
    key = cv2.waitKey(delay)
    if key == ord('q'):
        break

cv2.destroyAllWindows()

In [ ]:
#TODO: Use MOG2 to detect the moving objects in the video

history = 400  # Number of frames to use to build the background model (valores grandes: más estable pero tarda en adaptarse, valores pequeños: se adapta rápido pero es inestable)
varThreshold = 20  # Threshold to detect the background (umbral que determina si un pixel es fondo o no lo es)
detectShadows = True  # If True the algorithm detects the shadows

# Create the MOG2 object
mog2 = cv2.createBackgroundSubtractorMOG2(history, varThreshold, detectShadows)


In [ ]:
#TODO: Use a loop to detect the moving objects in the video using the MOG2 algorithm and 

# Create a folder to store the videos
output_folder = "../data/results"
if not os.path.exists(output_folder):
    os.makedirs(output_folder)

videoname = f'output_{history}_{varThreshold}_{detectShadows}.avi' # Name of the output video file with the parameters
videopath = os.path.join(output_folder, videoname)

# Create a VideoWriter object to save the video
fourcc = cv2.VideoWriter_fourcc(*"XVID") # Codec to use
frame_size = (frame_width, frame_height) # Size of the frames
fps = frame_rate # Frame rate of the video
out = cv2.VideoWriter(filename=videopath, fourcc=fourcc, fps=fps, frameSize=frame_size)

for frame in frames:
    # Apply the MOG2 algorithm to detect the moving objects
    mask = mog2.apply(frame)
    # Convert to BGR the mask to store it in the video
    mask = cv2.cvtColor(mask, cv2.COLOR_GRAY2BGR)
    # Save the mask in a video
    out.write(mask)

out.release()
print(f"Vídeo guardado en: {videopath}")

In [ ]:
#TODO: Use VideoCapture from OpenCV to read the video (slow_traffic_small.mp4)
videopath = "../data/partC/slow_traffic_small.mp4"  # Path to the video file
cap = cv2.VideoCapture(filename=videopath)  # Complete this line to read the video file

#TODO: Check if the video was successfully opened
if not cap.isOpened():
    print('Error: Could not open the video file')
    os._exit(0)

#TODO: Get the szie of frames and the frame rate of the video
frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
frame_rate = cap.get(cv2.CAP_PROP_FPS)

# Añadimos un print para mostrar las características del video con el que estamos trabajando
print("Video properties:")
print(" - Width :", frame_width)
print(" - Height:", frame_height)
print(" - FPS   :", frame_rate)

#TODO: Use a loop to read the frames of the video and store them in a list
frames = []
while True:
    ret, frame = cap.read() # Esto me permite leer cada uno de los frames que componen el video (al final el vídeo es una sucesión de imágenes)
    if not ret:
        break
    frames.append(frame)


cap.release()

# Define the parameters for Lucas-Kanade optical flow (probamos varios valores que hemos consultado en internet hasta conseguir unos adecuados)
winSize=(15, 15)
maxLevel=2
criteria= (cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 10, 0.03)

In [ ]:
#TODO: Detect the initial points of interest in the first frame

# Convert the first frame to grayscale
prev_gray = cv2.cvtColor(src=frames[0], code=cv2.COLOR_BGR2GRAY)

# Define the parameters of the Shi-Tomasi algorithm
mask = None  # sirve por si queremos restringir la detección a una región específica de la imagen
maxCorners = 200  # Número máximo de corners que queremos detectar
qualityLevel = 0.3  # Calidad mínima requerida para que se considere que un punto es "bueno" y lo detecte
minDistance = 7  # Distancia mínima entre puntos detectados
blockSize = 7  # Tamaño del vecindario usado para evaluar si el punto es corner o no

# Use the function goodFeaturesToTrack to detect the points of interest
p0 = cv2.goodFeaturesToTrack(prev_gray, mask=mask, maxCorners=maxCorners, qualityLevel=qualityLevel, minDistance=minDistance, blockSize=blockSize)

In [ ]:
#TODO: Use a loop to track the points of interest in the rest of the frames

# Create a mask image for drawing purposes
frame = frames[0]
mask = np.zeros_like(frame)

for i, frame in enumerate(frames[1:]):
    input_frame = frame
    # Convert the frame to grayscale
    frame_gray = cv2.cvtColor(input_frame, cv2.COLOR_BGR2GRAY)
    # Calculate the optical flow using the Lucas-Kanade algorithm

    p1, st, err = cv2.calcOpticalFlowPyrLK(prevImg=prev_gray, nextImg=frame_gray, prevPts=p0, nextPts=None)

    # Select the points that were successfully tracked
    good_new = p1[st == 1]
    good_old = p0[st == 1]

    # Draw the tracks
    for i, (new, old) in enumerate(zip(good_new, good_old)):
        a, b = new.ravel().astype(int)
        c, d = old.ravel().astype(int)
        input_frame = cv2.circle(input_frame, (a, b), 5, (0, 0, 255), -1)
        mask = cv2.line(mask, (a, b), (c, d), (0, 255, 0), 2)

    # Update the inputs for the next iteration
    prev_gray = frame_gray.copy()
    p0 = good_new.reshape(-1, 1, 2)
    
    # Show the frame with the tracks
    cv2.imshow('Visualizacion Flujo Optico', cv2.add(input_frame, mask))
    key = cv2.waitKey(1)
    if key == ord('q'):
        break

cv2.destroyAllWindows()

In [ ]:
#TODO: Use VideoCapture from OpenCV to read the video (slow_traffic_small.mp4)
videopath = "../data/partC/slow_traffic_small.mp4" # Path to the video file
cap = cv2.VideoCapture(videopath)  # Complete this line to read the video file
 
#TODO: Check if the video was successfully opened
if not cap.isOpened():
    print('Error: Could not open the video file')
 
#TODO: Get the szie of frames and the frame rate of the video
frame_width = cap.get(cv2.CAP_PROP_FRAME_WIDTH)
frame_height = cap.get(cv2.CAP_PROP_FRAME_HEIGHT)
frame_rate = cap.get(cv2.CAP_PROP_FPS)
 
#TODO: Use a loop to read the frames of the video and store them in a list
frames = []
while True:
    ret, frame = cap.read()
    if not ret:
        break
    frames.append(frame)
 
 
cap.release()


###############################################################################################
 
 
# TODO: Create the Kalman filter object
kf = cv2.KalmanFilter(4, 2) # 4 state variables, 2 measurement variables
# TODO: Initialize the state of the Kalman filter
kf.measurementMatrix = np.array([[1, 0, 0, 0], [0, 1, 0, 0]], np.float32) # Measurement matrix np.array of shape (2, 4) and type np.float32
kf.transitionMatrix = np.array([[1, 0, 1, 0], [0, 1, 0, 1], [0, 0, 1, 0], [0, 0, 0, 1]], np.float32) # Transition matrix np.array of shape (4, 4) and type np.float32
kf.processNoiseCov = np.eye(4, dtype=np.float32) * 0.01 # Process noise covariance np.array of shape (4, 4) and type np.float32
 
measurement = np.array((2, 1), np.float32)
prediction = np.zeros((2, 1), np.float32)
 
#TODO: Show the frames to select the initial position of the object
 
for i, frame in enumerate(frames):
    # Show the frame
    cv2.imshow('Frame', frame)
    # Wait for the key
    key = cv2.waitKey(0)
    print(key)
    # If the key is 'n' continue to the next frame
    if key == ord('n'):
        continue
    # If the key is 's' select the position of the object
    elif key == ord('s'):
        # Select the position of the object
        x, y, w, h = cv2.selectROI('Frame', frame, False)
        # x,y son las coordenadas de la esquina superior izquierda del ROI seleccionado
        track_window = (x, y, w, h)
        #TODO: Compute the center of the object
        cx = x + w // 2
        cy = y + h // 2
        print(f'Center of the object: {cx}, {cy}')
        #TODO: Initialize the state of the Kalman filter
        kf.statePost = np.array([[cx], [cy], [0], [0]], np.float32)
        # Initialize the covariance matrix
        kf.errorCovPost = np.eye(4, dtype=np.float32)
        #Predict the position of the object
        prediction = kf.predict()
 
        #TODO: Update the measurement and correct the Kalman filter
        measurement = np.array([[cx], [cy]], np.float32)
        kf.correct(measurement)
 
        #TODO: Crop the object
        crop = frame[y:y+h, x:x+w].copy()  # Recorta exactamente el ROI seleccionado
       
        hsv_crop = cv2.cvtColor(crop, cv2.COLOR_BGR2HSV)
        #TODO: Compute the histogram of the cropped object (Reminder: Use only the Hue channel (0-180))
        mask = cv2.inRange(hsv_crop, np.array([0, 50, 50]), np.array([180, 255, 255]))
        crop_hist = cv2.calcHist([hsv_crop], [0], mask=mask, histSize=[180], ranges=[0, 180])
        cv2.normalize(crop_hist, crop_hist, 0, 255, cv2.NORM_MINMAX)
       
        print(f'Initial position selected: {x}, {y}')
        break
 
cv2.destroyAllWindows()

In [ ]:
#TODO: Use the Kalman filter to predict the position of the points of interest
 
term_crit = (cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 30, 1)
 
for frame in frames[i:]:
    #TODO: Copy the frame
    input_frame = frame.copy()
    #TODO: Convert the frame to HSV
    img_hsv = cv2.cvtColor(input_frame, cv2.COLOR_BGR2HSV)
   
    # Compute the back projection of the histogram
    img_bproject = cv2.calcBackProject([img_hsv], [0], crop_hist, [0, 180], 1)
   
    # Apply the mean shift algorithm to the back projection
    ret, track_window = cv2.meanShift(img_bproject, track_window, term_crit)
    x_,y_,w_,h_ = track_window
    #TODO: Compute the center of the object
    c_x = x_ + w_ // 2
    c_y = y_ + h_ // 2
   
    # Predict the position of the object
    prediction = kf.predict()
 
    #TODO: Update the measurement and correct the Kalman filter
    measurement = np.array([[c_x], [c_y]], np.float32)
    kf.correct(measurement)
 
   
    # Draw the predicted position as RED rectangle (same size as original)
    pred_x = int(prediction[0][0] - w_ // 2)
    pred_y = int(prediction[1][0] - h_ // 2)
    cv2.rectangle(input_frame, (pred_x, pred_y), (pred_x + w_, pred_y + h_), (0, 0, 255), 2)
   
    # Draw the measured position as GREEN rectangle (same size as original)
    cv2.rectangle(input_frame, (x_, y_), (x_ + w_, y_ + h_), (0, 255, 0), 2)
   
    # También mantener los círculos (opcional)
    cv2.circle(input_frame, (int(prediction[0][0]), int(prediction[1][0])), 5, (0, 0, 255), -1)
    cv2.circle(input_frame, (int(c_x), int(c_y)), 5, (0, 255, 0), -1)
   
    # Show the frame with the predicted position
    cv2.imshow('Frame', input_frame)
    key = cv2.waitKey(0)
    if key == ord('q'):
        break
 
cv2.destroyAllWindows()

In [ ]:
cap.get(cv2.CAP_PROP_FRAME_WIDTH)
cap.get(cv2.CAP_PROP_FRAME_HEIGHT)
cap.get(cv2.CAP_PROP_FPS)

# APUNTES GENERALES

---


# Guía de Funciones OpenCV (cv2)

## 1. Carga y Visualización de Imágenes

### `cv2.imread()`
**Descripción:** Lee una imagen desde un archivo.

**Parámetros:**
- `filename` (str): Ruta del archivo de imagen
- `flags` (int, opcional): Modo de lectura
  - `cv2.IMREAD_COLOR` (1): Carga imagen en color (BGR) - por defecto
  - `cv2.IMREAD_GRAYSCALE` (0): Carga en escala de grises
  - `cv2.IMREAD_UNCHANGED` (-1): Carga imagen con canal alpha si existe

**Retorna:** Array numpy (imagen) o None si falla

**Ejemplo:**
```python
import cv2

# Leer imagen en color
img = cv2.imread('imagen.jpg')

# Leer en escala de grises
img_gray = cv2.imread('imagen.jpg', cv2.IMREAD_GRAYSCALE)
```

---

### `cv2.imshow()`
**Descripción:** Muestra una imagen en una ventana.

**Parámetros:**
- `winname` (str): Nombre de la ventana
- `mat` (numpy.ndarray): Imagen a mostrar

**Nota:** Requiere `cv2.waitKey()` para mostrar la ventana.

**Ejemplo:**
```python
cv2.imshow('Mi Imagen', img)
cv2.waitKey(0)  # Espera hasta pulsar tecla
cv2.destroyAllWindows()
```

---

### `cv2.imwrite()`
**Descripción:** Guarda una imagen en un archivo.

**Parámetros:**
- `filename` (str): Ruta donde guardar
- `img` (numpy.ndarray): Imagen a guardar
- `params` (list, opcional): Parámetros de compresión

**Retorna:** True si se guardó correctamente

**Ejemplo:**
```python
cv2.imwrite('salida.jpg', img)
cv2.imwrite('salida.png', img, [cv2.IMWRITE_PNG_COMPRESSION, 9])
```

---

## 2. Transformaciones de Color y Espacio

### `cv2.cvtColor()`
**Descripción:** Convierte una imagen de un espacio de color a otro.

**Parámetros:**
- `src` (numpy.ndarray): Imagen origen
- `code` (int): Código de conversión
  - `cv2.COLOR_BGR2GRAY`: BGR a escala de grises
  - `cv2.COLOR_BGR2HSV`: BGR a HSV
  - `cv2.COLOR_BGR2RGB`: BGR a RGB
  - `cv2.COLOR_GRAY2BGR`: Escala de grises a BGR

**Retorna:** Imagen convertida

**Ejemplo:**
```python
# Convertir a escala de grises
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

# Convertir a HSV
hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
```

---

### `cv2.inRange()`
**Descripción:** Crea una máscara binaria de píxeles que están dentro de un rango.

**Parámetros:**
- `src` (numpy.ndarray): Imagen origen
- `lowerb` (array): Límite inferior del rango
- `upperb` (array): Límite superior del rango

**Retorna:** Máscara binaria (0 o 255)

**Ejemplo:**
```python
hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)

# Detectar color rojo
lower_red = np.array([0, 100, 100])
upper_red = np.array([10, 255, 255])
mask = cv2.inRange(hsv, lower_red, upper_red)
```

---

## 3. Transformaciones Geométricas

### `cv2.resize()`
**Descripción:** Redimensiona una imagen.

**Parámetros:**
- `src` (numpy.ndarray): Imagen origen
- `dsize` (tuple): Tamaño de salida (ancho, alto). Si es (0,0), usa fx y fy
- `fx` (float, opcional): Factor de escala en x
- `fy` (float, opcional): Factor de escala en y
- `interpolation` (int, opcional): Método de interpolación
  - `cv2.INTER_LINEAR`: Interpolación bilineal (por defecto)
  - `cv2.INTER_NEAREST`: Vecino más cercano
  - `cv2.INTER_CUBIC`: Interpolación bicúbica

**Ejemplo:**
```python
# Redimensionar a tamaño específico
resized = cv2.resize(img, (640, 480))

# Redimensionar con factor de escala
resized = cv2.resize(img, None, fx=0.5, fy=0.5)
```

---

### `cv2.getRotationMatrix2D()`
**Descripción:** Calcula la matriz de transformación afín para rotación.

**Parámetros:**
- `center` (tuple): Centro de rotación (x, y)
- `angle` (float): Ángulo de rotación en grados (positivo = antihorario)
- `scale` (float): Factor de escala isométrico

**Retorna:** Matriz 2x3 de transformación

**Ejemplo:**
```python
height, width = img.shape[:2]
center = (width // 2, height // 2)

# Rotar 45 grados sin escalar
M = cv2.getRotationMatrix2D(center, 45, 1.0)
```

---

### `cv2.warpAffine()`
**Descripción:** Aplica una transformación afín a una imagen.

**Parámetros:**
- `src` (numpy.ndarray): Imagen origen
- `M` (numpy.ndarray): Matriz de transformación 2x3
- `dsize` (tuple): Tamaño de la imagen de salida (ancho, alto)
- `flags` (int, opcional): Método de interpolación
- `borderMode` (int, opcional): Método de extrapolación de bordes
- `borderValue` (scalar, opcional): Valor usado en bordes constantes

**Ejemplo:**
```python
height, width = img.shape[:2]
center = (width // 2, height // 2)
M = cv2.getRotationMatrix2D(center, 45, 1.0)

rotated = cv2.warpAffine(img, M, (width, height))
```

---

### `cv2.rotate()`
**Descripción:** Rota una imagen en múltiplos de 90 grados.

**Parámetros:**
- `src` (numpy.ndarray): Imagen origen
- `rotateCode` (int): Código de rotación
  - `cv2.ROTATE_90_CLOCKWISE`: 90° horario
  - `cv2.ROTATE_180`: 180°
  - `cv2.ROTATE_90_COUNTERCLOCKWISE`: 90° antihorario

**Ejemplo:**
```python
rotated = cv2.rotate(img, cv2.ROTATE_90_CLOCKWISE)
```

---

## 4. Operaciones con Máscaras y Bitwise

### `cv2.bitwise_and()`
**Descripción:** Operación AND bit a bit entre dos imágenes.

**Parámetros:**
- `src1` (numpy.ndarray): Primera imagen
- `src2` (numpy.ndarray): Segunda imagen
- `mask` (numpy.ndarray, opcional): Máscara de operación

**Ejemplo:**
```python
# Aplicar máscara a imagen
result = cv2.bitwise_and(img, img, mask=mask)
```

---

### `cv2.bitwise_or()`
**Descripción:** Operación OR bit a bit entre dos imágenes.

**Parámetros:**
- `src1` (numpy.ndarray): Primera imagen
- `src2` (numpy.ndarray): Segunda imagen

**Ejemplo:**
```python
combined = cv2.bitwise_or(img1, img2)
```

---

## 5. Calibración de Cámara

### `cv2.findChessboardCorners()`
**Descripción:** Encuentra las esquinas interiores de un patrón de tablero de ajedrez.

**Parámetros:**
- `image` (numpy.ndarray): Imagen en escala de grises
- `patternSize` (tuple): Número de esquinas interiores (columnas, filas)
- `flags` (int, opcional): Flags de detección

**Retorna:** (ret, corners)
- `ret` (bool): True si se encontraron todas las esquinas
- `corners` (numpy.ndarray): Array de coordenadas de esquinas

**Ejemplo:**
```python
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
ret, corners = cv2.findChessboardCorners(gray, (7, 6), None)

if ret:
    print("Esquinas encontradas")
```

---

### `cv2.cornerSubPix()`
**Descripción:** Refina las posiciones de las esquinas con precisión subpíxel.

**Parámetros:**
- `image` (numpy.ndarray): Imagen en escala de grises
- `corners` (numpy.ndarray): Posiciones iniciales de esquinas
- `winSize` (tuple): Mitad del tamaño de la ventana de búsqueda
- `zeroZone` (tuple): Mitad del tamaño de la zona muerta (-1,-1 para ninguna)
- `criteria` (tuple): Criterio de terminación

**Ejemplo:**
```python
criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, 0.001)
corners_refined = cv2.cornerSubPix(gray, corners, (11, 11), (-1, -1), criteria)
```

---

### `cv2.drawChessboardCorners()`
**Descripción:** Dibuja las esquinas detectadas del tablero.

**Parámetros:**
- `image` (numpy.ndarray): Imagen destino (será modificada)
- `patternSize` (tuple): Número de esquinas (columnas, filas)
- `corners` (numpy.ndarray): Posiciones de las esquinas
- `patternWasFound` (bool): Si se encontró el patrón completo

**Ejemplo:**
```python
cv2.drawChessboardCorners(img, (7, 6), corners, ret)
```

---

### `cv2.calibrateCamera()`
**Descripción:** Calibra una cámara usando múltiples vistas de un patrón de calibración.

**Parámetros:**
- `objectPoints` (list): Puntos 3D del patrón en coordenadas del mundo
- `imagePoints` (list): Puntos 2D correspondientes en las imágenes
- `imageSize` (tuple): Tamaño de la imagen (ancho, alto)
- `cameraMatrix` (numpy.ndarray): Matriz intrínseca inicial (puede ser None)
- `distCoeffs` (numpy.ndarray): Coeficientes de distorsión iniciales

**Retorna:** (ret, mtx, dist, rvecs, tvecs)
- `ret`: Error de reproyección
- `mtx`: Matriz de cámara
- `dist`: Coeficientes de distorsión
- `rvecs`: Vectores de rotación
- `tvecs`: Vectores de traslación

**Ejemplo:**
```python
ret, mtx, dist, rvecs, tvecs = cv2.calibrateCamera(
    objpoints, imgpoints, gray.shape[::-1], None, None
)
```

---

### `cv2.Rodrigues()`
**Descripción:** Convierte entre vector de rotación y matriz de rotación.

**Parámetros:**
- `src` (numpy.ndarray): Vector de rotación (3x1) o matriz de rotación (3x3)

**Retorna:** (dst, jacobian)
- Matriz de rotación si entrada es vector, o viceversa

**Ejemplo:**
```python
# Vector a matriz
rvec = np.array([0.1, 0.2, 0.3])
R, _ = cv2.Rodrigues(rvec)

# Matriz a vector
rvec, _ = cv2.Rodrigues(R)
```

---

## 6. Filtros y Procesamiento

### `cv2.filter2D()`
**Descripción:** Aplica un filtro/kernel personalizado a una imagen mediante convolución.

**Parámetros:**
- `src` (numpy.ndarray): Imagen origen
- `ddepth` (int): Profundidad de la imagen destino (-1 para usar la misma que src)
- `kernel` (numpy.ndarray): Kernel de convolución (matriz de floats)
- `anchor` (tuple, opcional): Posición del ancla en el kernel
- `delta` (float, opcional): Valor añadido al resultado

**Ejemplo:**
```python
# Kernel de detección de bordes
kernel = np.array([[-1, -1, -1],
                   [-1,  8, -1],
                   [-1, -1, -1]], dtype=np.float32)

filtered = cv2.filter2D(img, -1, kernel)
```

---

### `cv2.GaussianBlur()`
**Descripción:** Aplica un desenfoque Gaussiano.

**Parámetros:**
- `src` (numpy.ndarray): Imagen origen
- `ksize` (tuple): Tamaño del kernel (debe ser impar y positivo)
- `sigmaX` (float): Desviación estándar en X
- `sigmaY` (float, opcional): Desviación estándar en Y (si es 0, usa sigmaX)

**Ejemplo:**
```python
blurred = cv2.GaussianBlur(img, (5, 5), 0)

# Mayor desenfoque
blurred = cv2.GaussianBlur(img, (15, 15), 0)
```

---

### `cv2.threshold()`
**Descripción:** Aplica umbralización a una imagen.

**Parámetros:**
- `src` (numpy.ndarray): Imagen origen (típicamente en escala de grises)
- `thresh` (float): Valor umbral
- `maxval` (float): Valor máximo a usar con tipos THRESH_BINARY
- `type` (int): Tipo de umbralización
  - `cv2.THRESH_BINARY`: > thresh → maxval, sino → 0
  - `cv2.THRESH_BINARY_INV`: Inverso del anterior
  - `cv2.THRESH_OTSU`: Calcula umbral óptimo automáticamente

**Retorna:** (ret, thresh_img)

**Ejemplo:**
```python
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
ret, binary = cv2.threshold(gray, 127, 255, cv2.THRESH_BINARY)

# Umbralización automática con Otsu
ret, binary = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
```

---

### `cv2.Canny()`
**Descripción:** Detector de bordes de Canny.

**Parámetros:**
- `image` (numpy.ndarray): Imagen en escala de grises
- `threshold1` (float): Primer umbral para histéresis
- `threshold2` (float): Segundo umbral para histéresis
- `apertureSize` (int, opcional): Tamaño de apertura para Sobel (por defecto 3)
- `L2gradient` (bool, opcional): Si usar norma L2 (más precisa) o L1

**Ejemplo:**
```python
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
edges = cv2.Canny(gray, 100, 200)
```

---

### `cv2.dilate()`
**Descripción:** Aplica dilatación morfológica (expande regiones blancas).

**Parámetros:**
- `src` (numpy.ndarray): Imagen origen (binaria típicamente)
- `kernel` (numpy.ndarray): Elemento estructurante
- `iterations` (int, opcional): Número de veces que aplicar la dilatación

**Ejemplo:**
```python
kernel = np.ones((5, 5), np.uint8)
dilated = cv2.dilate(mask, kernel, iterations=1)
```

---

### `cv2.subtract()`
**Descripción:** Calcula la diferencia por elemento entre dos arrays.

**Parámetros:**
- `src1` (numpy.ndarray): Primera imagen
- `src2` (numpy.ndarray): Segunda imagen
- `mask` (numpy.ndarray, opcional): Máscara de operación

**Ejemplo:**
```python
diff = cv2.subtract(img1, img2)
```

---

### `cv2.absdiff()`
**Descripción:** Calcula la diferencia absoluta por elemento entre dos arrays.

**Parámetros:**
- `src1` (numpy.ndarray): Primera imagen
- `src2` (numpy.ndarray): Segunda imagen

**Ejemplo:**
```python
# Útil para detección de movimiento
diff = cv2.absdiff(frame1, frame2)
```

---

## 7. Detección de Características

### `cv2.cornerHarris()`
**Descripción:** Detector de esquinas de Harris.

**Parámetros:**
- `src` (numpy.ndarray): Imagen en escala de grises (float32)
- `blockSize` (int): Tamaño de la vecindad
- `ksize` (int): Tamaño de apertura del detector Sobel
- `k` (float): Parámetro libre de Harris (típicamente 0.04-0.06)

**Retorna:** Imagen con respuesta de esquinas (float32)

**Ejemplo:**
```python
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
gray = np.float32(gray)
corners = cv2.cornerHarris(gray, 2, 3, 0.04)

# Marcar esquinas fuertes
img[corners > 0.01 * corners.max()] = [0, 0, 255]
```

---

### `cv2.goodFeaturesToTrack()`
**Descripción:** Detecta esquinas fuertes en una imagen (método Shi-Tomasi).

**Parámetros:**
- `image` (numpy.ndarray): Imagen en escala de grises
- `maxCorners` (int): Número máximo de esquinas a retornar
- `qualityLevel` (float): Calidad mínima (0-1, típicamente 0.01)
- `minDistance` (float): Distancia mínima euclidiana entre esquinas
- `blockSize` (int, opcional): Tamaño de la vecindad

**Retorna:** Array de coordenadas de esquinas

**Ejemplo:**
```python
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
corners = cv2.goodFeaturesToTrack(gray, 100, 0.01, 10)

corners = np.int0(corners)
for corner in corners:
    x, y = corner.ravel()
    cv2.circle(img, (x, y), 3, (0, 255, 0), -1)
```

---

### `cv2.SIFT_create()` y `detectAndCompute()`
**Descripción:** Crea detector SIFT y detecta keypoints + descriptores.

**SIFT_create() - Sin parámetros obligatorios**

**detectAndCompute() - Parámetros:**
- `image` (numpy.ndarray): Imagen en escala de grises
- `mask` (numpy.ndarray, opcional): Máscara para detectar solo en ciertas regiones

**Retorna:** (keypoints, descriptors)

**Ejemplo:**
```python
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

sift = cv2.SIFT_create()
keypoints, descriptors = sift.detectAndCompute(gray, None)

# Dibujar keypoints
img_kp = cv2.drawKeypoints(gray, keypoints, None)
```

---

## 8. Dibujo

### `cv2.circle()`
**Descripción:** Dibuja un círculo en una imagen.

**Parámetros:**
- `img` (numpy.ndarray): Imagen (será modificada)
- `center` (tuple): Centro del círculo (x, y)
- `radius` (int): Radio
- `color` (tuple): Color BGR (ej: (255, 0, 0) para azul)
- `thickness` (int): Grosor del borde (-1 para rellenar)

**Ejemplo:**
```python
cv2.circle(img, (100, 100), 50, (0, 255, 0), 2)  # Círculo verde
cv2.circle(img, (200, 200), 30, (0, 0, 255), -1)  # Círculo rojo relleno
```

---

### `cv2.line()`
**Descripción:** Dibuja una línea en una imagen.

**Parámetros:**
- `img` (numpy.ndarray): Imagen (será modificada)
- `pt1` (tuple): Punto inicial (x, y)
- `pt2` (tuple): Punto final (x, y)
- `color` (tuple): Color BGR
- `thickness` (int, opcional): Grosor de la línea

**Ejemplo:**
```python
cv2.line(img, (0, 0), (500, 500), (255, 0, 0), 3)
```

---

### `cv2.drawMatches()`
**Descripción:** Dibuja las correspondencias encontradas entre dos imágenes.

**Parámetros:**
- `img1` (numpy.ndarray): Primera imagen
- `keypoints1` (list): Keypoints de la primera imagen
- `img2` (numpy.ndarray): Segunda imagen
- `keypoints2` (list): Keypoints de la segunda imagen
- `matches` (list): Lista de matches (DMatch objects)
- `outImg` (numpy.ndarray): Imagen de salida (puede ser None)
- `flags` (int, opcional): Flags de dibujo

**Ejemplo:**
```python
# Después de detectar SIFT y hacer matching
bf = cv2.BFMatcher()
matches = bf.knnMatch(desc1, desc2, k=2)

good_matches = []
for m, n in matches:
    if m.distance < 0.75 * n.distance:
        good_matches.append(m)

img_matches = cv2.drawMatches(img1, kp1, img2, kp2, good_matches, None)
```

---

## 9. Vídeo

### `cv2.VideoCapture()`
**Descripción:** Constructor para capturar vídeo de archivo o cámara.

**Parámetros:**
- `index` (int o str): Índice de cámara (0, 1, ...) o ruta de archivo

**Ejemplo:**
```python
# Capturar desde cámara
cap = cv2.VideoCapture(0)

# Capturar desde archivo
cap = cv2.VideoCapture('video.mp4')
```

---

### `VideoCapture.isOpened()`
**Descripción:** Verifica si la captura está inicializada correctamente.

**Retorna:** bool

**Ejemplo:**
```python
if not cap.isOpened():
    print("Error al abrir la cámara")
```

---

### `VideoCapture.get()`
**Descripción:** Obtiene propiedades de la captura.

**Parámetros:**
- `propId` (int): ID de la propiedad
  - `cv2.CAP_PROP_FRAME_WIDTH`: Ancho del frame
  - `cv2.CAP_PROP_FRAME_HEIGHT`: Alto del frame
  - `cv2.CAP_PROP_FPS`: Frames por segundo

**Ejemplo:**
```python
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)
```

---

### `VideoCapture.read()`
**Descripción:** Lee el siguiente frame del vídeo.

**Retorna:** (ret, frame)
- `ret` (bool): True si se leyó correctamente
- `frame` (numpy.ndarray): Frame capturado

**Ejemplo:**
```python
while True:
    ret, frame = cap.read()
    if not ret:
        break
    
    cv2.imshow('Frame', frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break
```

---

### `VideoCapture.release()`
**Descripción:** Libera la captura de vídeo.

**Ejemplo:**
```python
cap.release()
cv2.destroyAllWindows()
```

---

### `cv2.VideoWriter()`
**Descripción:** Constructor para escribir vídeo en archivo.

**Parámetros:**
- `filename` (str): Ruta del archivo de salida
- `fourcc` (int): Código de codec de 4 caracteres
- `fps` (float): Frames por segundo
- `frameSize` (tuple): Tamaño del frame (ancho, alto)
- `isColor` (bool, opcional): True para vídeo a color

**Ejemplo:**
```python
fourcc = cv2.VideoWriter_fourcc(*'XVID')
out = cv2.VideoWriter('output.avi', fourcc, 20.0, (640, 480))

while True:
    ret, frame = cap.read()
    if ret:
        out.write(frame)
    else:
        break

out.release()
```

---

### `cv2.createBackgroundSubtractorMOG2()`
**Descripción:** Crea un sustractor de fondo basado en mezcla de Gaussianas.

**Parámetros:**
- `history` (int, opcional): Longitud del historial
- `varThreshold` (float, opcional): Umbral de varianza
- `detectShadows` (bool, opcional): Si detectar sombras

**Retorna:** Objeto BackgroundSubtractor

**Ejemplo:**
```python
bg_subtractor = cv2.createBackgroundSubtractorMOG2(detectShadows=True)

while True:
    ret, frame = cap.read()
    if not ret:
        break
    
    fg_mask = bg_subtractor.apply(frame)
    cv2.imshow('Foreground', fg_mask)
```

---

## 10. Seguimiento y Flujo Óptico

### `cv2.calcOpticalFlowPyrLK()`
**Descripción:** Calcula flujo óptico usando el método Lucas-Kanade piramidal.

**Parámetros:**
- `prevImg` (numpy.ndarray): Primera imagen en escala de grises
- `nextImg` (numpy.ndarray): Segunda imagen en escala de grises
- `prevPts` (numpy.ndarray): Puntos a seguir en primera imagen (float32)
- `nextPts` (numpy.ndarray o None): Estimación inicial de puntos (puede ser None)
- `winSize` (tuple, opcional): Tamaño de ventana de búsqueda
- `maxLevel` (int, opcional): Número de niveles piramidales

**Retorna:** (nextPts, status, err)
- `nextPts`: Nuevas posiciones de puntos
- `status`: Array indicando si se encontró el punto (1) o no (0)
- `err`: Error de medida

**Ejemplo:**
```python
old_gray = cv2.cvtColor(old_frame, cv2.COLOR_BGR2GRAY)
p0 = cv2.goodFeaturesToTrack(old_gray, 100, 0.3, 7)

frame_gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
p1, st, err = cv2.calcOpticalFlowPyrLK(old_gray, frame_gray, p0, None)

# Seleccionar buenos puntos
good_new = p1[st == 1]
good_old = p0[st == 1]
```

---

### `cv2.selectROI()`
**Descripción:** Permite seleccionar una región de interés interactivamente.

**Parámetros:**
- `windowName` (str): Nombre de la ventana
- `img` (numpy.ndarray): Imagen donde seleccionar
- `showCrosshair` (bool, opcional): Mostrar cruz central
- `fromCenter` (bool, opcional): Dibujar desde el centro

**Retorna:** (x, y, w, h) - Coordenadas y dimensiones del ROI

**Ejemplo:**
```python
roi = cv2.selectROI('Select ROI', frame, False, False)
x, y, w, h = roi

cropped = frame[y:y+h, x:x+w]
```

---

### `cv2.calcBackProject()`
**Descripción:** Calcula la retroproyección de un histograma (útil para seguimiento).

**Parámetros:**
- `images` (list): Lista de imágenes
- `channels` (list): Lista de canales a usar
- `hist` (numpy.ndarray): Histograma a retroproyectar
- `ranges` (list): Rangos de valores para cada canal
- `scale` (float): Factor de escala

**Retorna:** Imagen de retroproyección

**Ejemplo:**
```python
hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
roi_hist = cv2.calcHist([roi_hsv], [0], None, [180], [0, 180])
cv2.normalize(roi_hist, roi_hist, 0, 255, cv2.NORM_MINMAX)

dst = cv2.calcBackProject([hsv], [0], roi_hist, [0, 180], 1)
```

---

## 11. Filtro de Kalman

### `cv2.KalmanFilter()`
**Descripción:** Constructor del filtro de Kalman.

**Parámetros:**
- `dynamParams` (int): Dimensión del vector de estado
- `measureParams` (int): Dimensión del vector de medida
- `controlParams` (int, opcional): Dimensión del vector de control

**Ejemplo:**
```python
kalman = cv2.KalmanFilter(4, 2)  # 4 estados, 2 medidas

# Configurar matrices
kalman.measurementMatrix = np.array([[1, 0, 0, 0],
                                      [0, 1, 0, 0]], np.float32)
kalman.transitionMatrix = np.array([[1, 0, 1, 0],
                                     [0, 1, 0, 1],
                                     [0, 0, 1, 0],
                                     [0, 0, 0, 1]], np.float32)
```

---

### `KalmanFilter.predict()`
**Descripción:** Calcula el estado predicho.

**Parámetros:**
- `control` (numpy.ndarray, opcional): Vector de control

**Retorna:** Estado predicho

**Ejemplo:**
```python
prediction = kalman.predict()
```

---

### `KalmanFilter.correct()`
**Descripción:** Actualiza el estado con una nueva medida.

**Parámetros:**
- `measurement` (numpy.ndarray): Vector de medida

**Retorna:** Estado corregido

**Ejemplo:**
```python
measurement = np.array([[x], [y]], np.float32)
estimated = kalman.correct(measurement)

# Usar predicción y corrección en bucle
while True:
    prediction = kalman.predict()
    # ... obtener medida real ...
    estimated = kalman.correct(measurement)
```

---

## Notas Importantes

### Tipos de Datos Comunes:
- **Imágenes BGR**: Array numpy uint8, shape (height, width, 3)
- **Imágenes en escala de grises**: Array numpy uint8, shape (height, width)
- **Máscaras binarias**: Array numpy uint8, valores 0 o 255
- **Imágenes float**: Array numpy float32 o float64, típicamente normalizadas [0, 1]
- **Matrices de transformación**: Array numpy float32 o float64

### Convenciones:
- OpenCV usa **BGR** en lugar de RGB por defecto
- Las coordenadas son **(x, y)** donde x es horizontal, y es vertical
- Los tamaños son **(ancho, alto)** no (alto, ancho)
- Las imágenes son arrays con shape **(alto, ancho, canales)**

### Gestión de Recursos:
```python
# Siempre liberar recursos de vídeo
cap.release()
out.release()
cv2.destroyAllWindows()
```